# Colon Xenium QC - Region_1

## Goal

Run reproducible, non-destructive technical QC for **Region_1**. This notebook is one of six structurally identical region notebooks; its region identifier is locked below. Full-data results must be produced on HPC. Local runs use a deterministic small subset only and cannot establish transcript-level or final slide readiness.

## Setup

The cell below defines HPC defaults and accepts explicit environment overrides for local subset validation. It refuses writable paths outside `colon_analysis`. The pipeline installs nothing; missing packages cause a clear preflight stop.

In [1]:
REGION_ID <- "Region_1"
EXECUTION_MODE <- toupper(Sys.getenv("COLON_QC_MODE", "FULL_HPC"))
stopifnot(EXECUTION_MODE %in% c("FULL_HPC", "LOCAL_SUBSET"))
PROJECT_ROOT <- Sys.getenv("COLON_PROJECT_ROOT", "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium")
PIPELINE_REPO <- Sys.getenv("COLON_PIPELINE_REPO", file.path(PROJECT_ROOT, "colon_analysis", "YNH_Xenium_Colon"))
INPUT_ROOT <- Sys.getenv("COLON_INPUT_ROOT", file.path(PROJECT_ROOT, "adipose_data_B2/20260814__101613__260814_Yanan_repeat/"))
RUN_LABEL <- Sys.getenv("COLON_RUN_LABEL", "colon_qc_hpc")
RUN_ROOT <- Sys.getenv("COLON_RUN_ROOT", file.path(PROJECT_ROOT, "colon_analysis", "colon_qc_outputs", RUN_LABEL))
TEMP_ROOT <- Sys.getenv("COLON_TEMP_ROOT", file.path(PROJECT_ROOT, "colon_analysis", "tmp", RUN_LABEL))
dir.create(TEMP_ROOT, recursive = TRUE, showWarnings = FALSE)
Sys.setenv(TMPDIR = TEMP_ROOT, TMP = TEMP_ROOT, TEMP = TEMP_ROOT)
source(file.path(PIPELINE_REPO, "R", "source.R"))
set.seed(20260814L)

### Key assumptions

All six regions use one installed Xenium panel. Mouse is the biological replicate; this notebook performs technical QC only. The primary cell cohort is prespecified as `5 < nFeature_Xenium < 200` and `10 < nCount_Xenium < 1000`. Segmentation, nucleus, area, control, and spatial findings remain separate review flags.

In [2]:
runtime <- validate_runtime_paths(project_root = PROJECT_ROOT, input_root = INPUT_ROOT, output_root = RUN_ROOT, temp_root = TEMP_ROOT)
regions <- discover_xenium_sections(INPUT_ROOT, expected_section_count = 6L)
stopifnot(identical(regions$region_id, expected_colon_regions()))
manifest <- utils::read.delim(file.path(PIPELINE_REPO, "config", "colon_sample_manifest.tsv"), check.names = FALSE)
validate_sample_manifest(manifest, expected_colon_regions())
fixed_thresholds <- read_fixed_cell_qc_thresholds(file.path(PIPELINE_REPO, "config", "fixed_cell_qc_thresholds.tsv"))
manifest[manifest$region_id == REGION_ID, , drop = FALSE]

$valid
[1] TRUE

$issues
character(0)

,tissue,region_id,mouse_id,position,section_id,biological_replicate_id,technical_replicate_id,genotype,treatment,condition,age_weeks,metadata_status,do_not_interpret,notes
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>
1,colon,Region_1,Mouse_1,top,Region_1,Mouse_1,Region_1,WT,Not_provided,Not_provided,Not_provided,VERIFIED_USER_SUPPLIED,FALSE,Mouse 1 top colon section


## Inputs and integrity

Discover the region directory from its identifier, inventory required files, and verify matrix/metadata alignment before loading counts. Raw data are read-only. Full transcript Parquet is not collected into memory; full-HPC transcript summaries use Arrow-backed aggregation.

In [3]:
region_record <- discover_one_section(INPUT_ROOT, REGION_ID)
region_dir <- region_record$region_dir[[1L]]
inventory <- inventory_section_files(region_dir, REGION_ID, calculate_md5 = FALSE)
integrity <- validate_section_integrity(region_dir, REGION_ID)
stopifnot(all(inventory$exists), isTRUE(integrity$dimension_match[[1L]]))
bundle <- import_xenium_mex(region_dir)
stopifnot(inherits(bundle$counts, "sparseMatrix"), identical(colnames(bundle$counts), bundle$cells$cell_id))

## Cell QC

Compute targeted-panel complexity, control burden, and segmentation review fields without deleting cells. Exact boundary values 5/200 features and 10/1000 counts fail. `primary_include` contains only the fixed rule; `strict_include` additionally excludes all review flags.

In [4]:
cell_qc <- calculate_xenium_cell_qc(bundle$counts, bundle$cells, REGION_ID, fixed_thresholds)
masks <- build_cell_downstream_masks(cell_qc$cell_metadata, provenance = paste(RUN_LABEL, EXECUTION_MODE, sep = "::"))
stopifnot(nrow(masks) == ncol(bundle$counts), all(!masks$strict_include | masks$primary_include))
table(primary_include = masks$primary_include, strict_include = masks$strict_include)

               strict_include
primary_include  FALSE   TRUE
          FALSE   5240      0
          TRUE   29318 115938

In [5]:
names(cell_qc)

[1] "cell_metadata" "thresholds"    "summary"

In [6]:
cell_qc$thresholds

,region_id,metric,lower,upper,value,method
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
1,Region_1,nCount_Xenium,10,1000.0000,285.00000,fixed_exclusive_user_approved
2,Region_1,nFeature_Xenium,5,200.0000,76.00000,fixed_exclusive_user_approved
method,Region_1,cell_area,0,211.1664,52.83281,median_minus_5MAD_plus_5MAD
11,Region_1,control_fraction_cell,0,0.0500,0.00000,max_5pct_or_q99.5


In [7]:
cell_qc$summary

region_id,input_cells,core_qc_pass,core_qc_fail,review_flagged,nucleus_missing,multiple_nuclei,segmentation_multiplet,area_outlier,high_control,cells_deleted
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
Region_1,150496,145256,5240,34558,25195,4628,4906,2162,6,0


In [8]:
head(cell_qc$cell_metadata, n=2)

,cell_id,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,⋯,nCount_Xenium,nFeature_Xenium,control_fraction_cell,nucleus_missing_flag,multiple_nuclei_flag,cell_area_outlier_flag,high_control_flag,segmentation_multiplet_flag,qc_core_pass,qc_review_flag
,<chr>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
aaaadcpb-1,aaaadcpb-1,384.9233,1344.709,456,0,0,0,0,0,456,⋯,456,90,0,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE
aaaaijik-1,aaaaijik-1,366.8179,1354.152,253,0,0,0,0,0,253,⋯,253,97,0,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE


In [9]:
colnames(cell_qc$cell_metadata)

[1] "cell_id"                     "x_centroid"                 
 [3] "y_centroid"                  "transcript_counts"          
 [5] "control_probe_counts"        "genomic_control_counts"     
 [7] "control_codeword_counts"     "unassigned_codeword_counts" 
 [9] "deprecated_codeword_counts"  "total_counts"               
[11] "cell_area"                   "nucleus_area"               
[13] "nucleus_count"               "segmentation_method"        
[15] "region_id"                   "nCount_Xenium"              
[17] "nFeature_Xenium"             "control_fraction_cell"      
[19] "nucleus_missing_flag"        "multiple_nuclei_flag"       
[21] "cell_area_outlier_flag"      "high_control_flag"          
[23] "segmentation_multiplet_flag" "qc_core_pass"               
[25] "qc_review_flag"

In [10]:
dim(cell_qc$cell_metadata)

[1] 150496     25

In [11]:
colnames(masks)

[1] "cell_id"                       "x_centroid"                   
 [3] "y_centroid"                    "transcript_counts"            
 [5] "control_probe_counts"          "genomic_control_counts"       
 [7] "control_codeword_counts"       "unassigned_codeword_counts"   
 [9] "deprecated_codeword_counts"    "total_counts"                 
[11] "cell_area"                     "nucleus_area"                 
[13] "nucleus_count"                 "segmentation_method"          
[15] "region_id"                     "nCount_Xenium"                
[17] "nFeature_Xenium"               "control_fraction_cell"        
[19] "nucleus_missing_flag"          "multiple_nuclei_flag"         
[21] "cell_area_outlier_flag"        "high_control_flag"            
[23] "segmentation_multiplet_flag"   "qc_core_pass"                 
[25] "qc_review_flag"                "primary_include"              
[27] "strict_include"                "hotspot_review_cell"          
[29] "hotspot_sensitivity_include"   "section_status"               
[31] "mask_rule_primary"             "mask_rule_strict"             
[33] "mask_rule_hotspot_sensitivity" "provenance"

In [12]:
dim(masks)

[1] 150496     34

## Outputs and checks

Write an auditable region bundle beneath the run root: sparse raw counts, all cell metadata and masks, thresholds, inventory/integrity, run configuration, readiness evidence, and session information. The source notebook stays output-free; executed copies belong under the run directory.

In [13]:
section_output_dir <- file.path(RUN_ROOT, "sections", REGION_ID)
result <- write_colon_region_qc_bundle(
  project_root = PROJECT_ROOT, output_dir = section_output_dir, region_id = REGION_ID,
  run_label = RUN_LABEL, execution_mode = EXECUTION_MODE, manifest = manifest,
  inventory = inventory, integrity = integrity, bundle = bundle, cell_qc = cell_qc, masks = masks
)
stopifnot(validate_colon_region_qc_bundle(section_output_dir, REGION_ID))
result

region_id,output_dir,cells,primary_include,cells_deleted
<chr>,<chr>,<int>,<int>,<int>
Region_1,/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu/YNH_Xenium/colon_analysis/colon_qc_outputs/colon_qc_hpc/sections/Region_1,150496,145256,0


## Next steps

Review region tables and figures, especially assignment/control burden, boundary counts, segmentation flags, and spatial hotspots. A `PASS` here is technical evidence only. After all six regions finish under the same run label and mode, execute `02_slide_QC_summary.ipynb`. Full-data HPC output - not this local subset - determines final readiness.